# Kursus 2: Supervised Machine Learning: Regression

## Bagian 2e: Lab Regularisasi

> ### 📝 Catatan Belajar: Lab Regularisasi
>
> **Dataset:** `boston_housing_clean.pickle` (data Boston Housing)
>
> **Yang dipelajari:**
> - Standardisasi data dengan `StandardScaler` dan membandingkan koefisien dengan dan tanpa scaling.
> - Memakai Lasso dengan `alpha` berbeda dan membandingkan jumlah koefisien nol serta total magnitudonya.
> - Menghitung R² dengan train/test split, di mana scaling hanya di-fit pada data latih.
> - Membandingkan regularisasi L1 (Lasso) dan L2 (Ridge) pada `alpha` yang sama.

## Tujuan Pembelajaran

Setelah pelajaran ini, kamu diharapkan mampu:

* Menerapkan standardisasi data
* Menerapkan berbagai varian regresi dengan regularisasi
* Menggabungkan standardisasi data dengan prosedur train-test split
* Menerapkan regularisasi untuk mencegah overfitting pada masalah regresi

In [1]:
import numpy as np
import pandas as pd
from helper import boston_dataframe

np.set_printoptions(precision=3, suppress=True)

### Memuat Data Boston

**Catatan:** Lihat file `helper.py` untuk melihat bagaimana data Boston dibaca.

In [2]:
boston = boston_dataframe(description=True)
boston_data = boston[0]
boston_description = boston[1]

In [3]:
boston_data

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0.0,0.573,6.593,69.1,2.4786,1.0,273.0,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0.0,0.573,6.120,76.7,2.2875,1.0,273.0,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0.0,0.573,6.976,91.0,2.1675,1.0,273.0,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0.0,0.573,6.794,89.3,2.3889,1.0,273.0,21.0,393.45,6.48,22.0


## Standardisasi Data

**Standardisasi** data berarti mentransformasi setiap variabel agar lebih mendekati distribusi normal **standar**, dengan mean 0 dan standar deviasi 1.

Objek [`StandardScaler`](http://scikit-learn.org/dev/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler) pada SciKit Learn dapat melakukannya.

**Buat X dan y**:

In [4]:
y_col = "MEDV"

X = boston_data.drop(y_col, axis=1)
y = boston_data[y_col]

**Impor, fit, dan transform menggunakan `StandardScaler`**

In [5]:
from sklearn.preprocessing import StandardScaler

s = StandardScaler()
X_ss = s.fit_transform(X)

### Latihan: 

Pastikan standard scaling sudah benar

Petunjuk:

In [6]:
a = np.array([[1, 2, 3], 
              [4, 5, 6]]) 
print(a) # 2 rows, 3 columns

[[1 2 3]
 [4 5 6]]


In [32]:
a.mean(axis=0) # mean along the *columns*

array([2.5, 3.5, 4.5])

In [33]:
a.mean(axis=1) # mean along the *rows*

array([2., 5.])

In [34]:
### BEGIN SOLUTION
X2 = np.array(X)
man_transform = (X2-X2.mean(axis=0))/X2.std(axis=0)
np.allclose(man_transform, X_ss)
### END SOLUTION

True

### Koefisien dengan dan tanpa scaling

In [35]:
from sklearn.linear_model import LinearRegression

In [36]:
lr = LinearRegression()

y_col = "MEDV"

X = boston_data.drop(y_col, axis=1)
y = boston_data[y_col]

In [37]:
lr.fit(X, y)
print(lr.coef_) # min = -18

[ -0.108   0.046   0.021   2.687 -17.767   3.81    0.001  -1.476   0.306
  -0.012  -0.953   0.009  -0.525]


#### Diskusi (bersama): 

Koefisien berada pada skala yang sangat berbeda. Apakah ini "buruk"?

In [38]:
from sklearn.preprocessing import StandardScaler

In [39]:
s = StandardScaler()
X_ss = s.fit_transform(X)

In [15]:
lr2 = LinearRegression()
lr2.fit(X_ss, y)
print(lr2.coef_) # coefficients now "on the same scale"

[-0.928  1.082  0.141  0.682 -2.057  2.674  0.019 -3.104  2.662 -2.077
 -2.061  0.849 -3.744]


### Latihan: 

Berdasarkan hasil ini, fitur mana yang paling "berpengaruh" (pertanyaan ini sengaja dibuat agak ambigu)? "Ke arah mana" fitur itu memengaruhi "y"?

**Petunjuk:** Ingat dari minggu lalu bahwa kita bisa "menggabungkan" nama fitur sebuah DataFrame `df` dengan model `model` yang dilatih pada DataFrame tersebut memakai:

```python
dict(zip(df.columns.values, model.coef_))
```

In [41]:
### BEGIN SOLUTION
pd.DataFrame(zip(X.columns, lr2.coef_)).sort_values(by=1)
### END SOLUTION

,0,1
12,LSTAT,-3.743627
7,DIS,-3.104044
9,TAX,-2.076782
10,PTRATIO,-2.060607
4,NOX,-2.056718
0,CRIM,-0.928146
6,AGE,0.019466
2,INDUS,0.140900
3,CHAS,0.681740
11,B,0.849268


Jika hanya melihat besar koefisien yang sudah distandardisasi, LSTAT, DIS, RM, dan RAD adalah yang paling 'berpengaruh'. Sklearn tidak menyediakan signifikansi statistik bawaan untuk setiap variabel ini, yang bisa memperkuat atau memperlemah klaim tersebut.

### Lasso dengan dan tanpa scaling

Kita sudah membahas Lasso di kuliah. 

Mari kita tinjau bersama:

1. Apa bedanya Lasso dengan Regresi Linear biasa?
1. Apakah standardisasi lebih penting atau kurang penting pada Lasso dibanding Regresi Linear? Mengapa?

In [18]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import PolynomialFeatures

#### Membuat fitur polinomial

[`PolynomialFeatures`](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)

In [48]:
pf = PolynomialFeatures(degree=2, include_bias=False,)
X_pf = pf.fit_transform(X)


**Catatan:** Kita memakai `include_bias=False` karena `Lasso` sudah menyertakan bias secara default.

In [49]:
X_pf_ss = s.fit_transform(X_pf)

### Lasso

[Dokumentasi `Lasso`](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)

In [50]:
las = Lasso()
las.fit(X_pf_ss, y)
las.coef_ 

array([-0.   ,  0.   , -0.   ,  0.   , -0.   ,  0.   , -0.   , -0.   ,
       -0.   , -0.   , -0.991,  0.   , -0.   , -0.   ,  0.   , -0.   ,
        0.068, -0.   , -0.   , -0.   , -0.   , -0.   , -0.   , -0.   ,
       -0.   , -0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,
        0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.   ,  0.   ,
       -0.   , -0.   , -0.   , -0.05 , -0.   , -0.   , -0.   , -0.   ,
       -0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,
        0.   ,  0.   ,  0.   , -0.   , -0.   , -0.   , -0.   , -0.   ,
       -0.   , -0.   ,  0.   , -0.   ,  3.3  , -0.   , -0.   , -0.   ,
       -0.   , -0.   ,  0.42 , -3.498, -0.   , -0.   , -0.   , -0.   ,
       -0.   ,  0.   , -0.   , -0.   , -0.   , -0.146, -0.   , -0.   ,
       -0.   , -0.   , -0.   , -0.   , -0.   , -0.   , -0.   , -0.   ,
       -0.   , -0.   , -0.   ,  0.   , -0.   ,  0.   , -0.   , -0.   ])

### Latihan

Bandingkan 

* Jumlah besar (magnitudo) koefisien
* Jumlah koefisien yang bernilai nol

untuk Lasso dengan alpha 0,1 vs. 1.

Sebelum mengerjakan latihan, jawab pertanyaan berikut masing-masing dalam satu kalimat:

* Mana yang diperkirakan memiliki magnitudo lebih besar?
* Mana yang diperkirakan memiliki lebih banyak nol?

In [51]:
### BEGIN SOLUTION
las01 = Lasso(alpha = 0.1)
las01.fit(X_pf_ss, y)
print('sum of coefficients:', abs(las01.coef_).sum() )
print('number of coefficients not equal to 0:', (las01.coef_!=0).sum())

sum of coefficients: 26.138682362877965
number of coefficients not equal to 0: 23


In [52]:
las1 = Lasso(alpha = 1)
las1.fit(X_pf_ss, y)
print('sum of coefficients:',abs(las1.coef_).sum() )
print('number of coefficients not equal to 0:',(las1.coef_!=0).sum())
### END SOLUTION

sum of coefficients: 8.47240504455307
number of coefficients not equal to 0: 7


Dengan regularisasi yang lebih besar (alpha lebih tinggi), penalti untuk bobot yang besar akan lebih besar sehingga koefisien terdorong mengecil. Jadi alpha yang lebih tinggi berarti magnitudo lebih kecil dan lebih banyak koefisien yang terdorong menjadi 0.

### Latihan: $R^2$

Hitung $R^2$ dari setiap model tanpa train/test split.

Ingat bahwa kita mengimpor $R^2$ dengan:

```python
from sklearn.metrics import r2_score
```

In [30]:
### BEGIN SOLUTION
from sklearn.metrics import r2_score
r2_score(y,las1.predict(X_pf_ss))
### END SOLUTION

0.7207000417838496

In [31]:
### BEGIN SOLUTION
from sklearn.metrics import r2_score
r2_score(y,las01.predict(X_pf_ss))
### END SOLUTION

0.8366045990855127

#### Diskusikan:

Apakah regularisasi pernah meningkatkan performa model jika kita mengevaluasi pada dataset yang sama dengan data latih?

## Dengan train/test split

#### Diskusikan

Apakah ada masalah pada apa yang sudah kita lakukan sejauh ini?

**Petunjuk:** Pikirkan cara kita melakukan scaling fitur.

Diskusikan dalam kelompok dua atau tiga orang.

In [53]:
from sklearn.model_selection import train_test_split

In [54]:
X_train, X_test, y_train, y_test = train_test_split(X_pf, y, test_size=0.3, 
                                                    random_state=72018)

In [55]:
X_train_s = s.fit_transform(X_train)
las.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred = las.predict(X_test_s)
r2_score(y_pred, y_test)

0.33177406838134416

In [57]:
X_train_s = s.fit_transform(X_train)
las1.fit(X_train_s, y_train)
X_test_s = s.fit_transform(X_test)
y_pred = las1.predict(X_test_s)
r2_score(y_pred, y_test)

0.34947124693080767

In [58]:
X_train_s = s.fit_transform(X_train)
las01.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred = las01.predict(X_test_s)
r2_score(y_pred, y_test)

0.7102444090776476

### Latihan


#### Bagian 1:

Lakukan hal yang sama dengan Lasso dengan:

* `alpha` sebesar 0,001
* Naikkan `max_iter` menjadi 100000 untuk memastikan konvergensi. 

Hitung $R^2$ dari model tersebut.

Silakan salin kode dari atas, tetapi tulis komentar satu kalimat di atas setiap baris kode yang menjelaskan alasan melakukannya.

#### Bagian 2:

Lakukan prosedur yang sama seperti sebelumnya, tetapi dengan Regresi Linear.

Hitung $R^2$ dari model ini.

#### Bagian 3: 

Bandingkan jumlah nilai absolut koefisien kedua model, serta jumlah koefisien yang bernilai nol. Berdasarkan ukuran-ukuran ini, model mana yang merupakan deskripsi yang "lebih sederhana" tentang hubungan antara fitur dan target?

In [59]:
### BEGIN SOLUTION

# Part 1

# Decreasing regularization and ensuring convergence
las001 = Lasso(alpha = 0.001, max_iter=100000)

# Transforming training set to get standardized units
X_train_s = s.fit_transform(X_train)

# Fitting model to training set
las001.fit(X_train_s, y_train)

# Transforming test set using the parameters defined from training set
X_test_s = s.transform(X_test)

# Finding prediction on test set
y_pred = las001.predict(X_test_s)

# Calculating r2 score
print("r2 score for alpha = 0.001:", r2_score(y_pred, y_test))


# Part 2

# Using vanilla Linear Regression
lr = LinearRegression()

# Fitting model to training set
lr.fit(X_train_s, y_train)

# predicting on test set
y_pred_lr = lr.predict(X_test_s)

# Calculating r2 score
print("r2 score for Linear Regression:", r2_score(y_pred_lr, y_test))


# Part 3
print('Magnitude of Lasso coefficients:', abs(las001.coef_).sum())
print('Number of coeffients not equal to 0 for Lasso:', (las001.coef_!=0).sum())

print('Magnitude of Linear Regression coefficients:', abs(lr.coef_).sum())
print('Number of coeffients not equal to 0 for Linear Regression:', (lr.coef_!=0).sum())
### END SOLUTION

r2 score for alpha = 0.001: 0.8686454101886707
r2 score for Linear Regression: 0.8555202098064174
Magnitude of Lasso coefficients: 436.26164263065647
Number of coeffients not equal to 0 for Lasso: 89
Magnitude of Linear Regression coefficients: 1185.285825446829
Number of coeffients not equal to 0 for Linear Regression: 104


## Regularisasi L1 vs. L2

Seperti disebutkan di slide: regresi `Lasso` dan `Ridge` memiliki sintaks yang sama di SciKit Learn.

Sekarang kita bandingkan hasil regresi Ridge vs. Lasso:

[`Ridge`](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)

In [60]:
from sklearn.linear_model import Ridge

### Latihan

Dengan mengikuti dokumentasi Ridge di atas:

1. Buat objek Ridge `r` dengan `alpha` yang sama seperti `las001`.
2. Fit objek tersebut pada `X` dan `y` lalu cetak koefisien yang dihasilkan.

In [61]:
### BEGIN SOLUTION
# Decreasing regularization and ensuring convergence
r = Ridge(alpha = 0.001)
X_train_s = s.fit_transform(X_train)
r.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred_r = r.predict(X_test_s)

# Calculating r2 score
r.coef_
### END SOLUTION

array([  8.727,  10.354, -24.683,   5.299,  -3.13 ,  14.781,  22.301,
       -23.483,  27.356,  -1.512,  16.949,  22.531,  11.444,   1.055,
         0.437,  14.259,   1.83 ,  -8.737,   4.726,  -3.748,  -3.557,
       -16.521,  -6.918,   6.064,  -1.363,   4.604,  -1.303,  -0.057,
        -0.301, -12.828,   2.016,   1.084,  -0.655,  -1.092,   4.458,
        -4.247,   4.156,  -1.282,   8.71 ,  -0.285,  -8.595,  11.595,
         6.544,   1.271,   1.827,   1.007,   1.554,   5.173,  -4.685,
         5.299,  -3.199,  -8.715,   0.983,   1.119,   0.284,  -1.683,
        -2.884,   2.985,  -0.756,  12.297,   0.931,  -7.594,  18.292,
       -22.124,  35.725, -21.894,  -7.789,   1.774,   4.101, -12.263,
        -3.741,  -5.525, -16.111,  -5.706,  -4.666,  -9.884,   0.989,
        -0.092,  17.145, -14.214,  -2.955,  -2.785,  -5.404,  11.825,
         0.085,  -4.352,  -5.616,  -2.843,   0.826, -29.129,  49.636,
       -21.571,  -1.263,  -8.811, -15.822,  11.975,  -1.023,   2.911,
        -0.656,  -4.

In [62]:
las001 # same alpha as Ridge above

Lasso(alpha=0.001, max_iter=100000)

In [63]:
las001.coef_

array([  0.   ,   0.   , -16.945,   2.562,   0.   ,  13.392,   9.907,
       -19.96 ,   9.039,   0.   ,   6.138,  17.628,  11.446,   1.21 ,
         0.226,  11.578,   2.197,  -7.397,   4.117,  -1.589,  -2.157,
        -9.837,   0.   ,   0.   ,  -1.089,   3.793,   0.389,   0.2  ,
        -0.297,  -3.352,   0.396,   0.719,  -0.756,  -0.753,   2.388,
        -0.845,   2.524,  -0.973,   3.882,  -0.983,   6.609,   6.138,
         4.201,   0.858,   1.96 ,   2.661,  -4.148,   2.937,  -4.556,
         2.112,  -1.983,  -7.439,   1.607,   1.668,  -1.262,  -0.   ,
        -0.   ,   2.622,  -0.861,   3.353,  -0.   ,  -2.451,   9.171,
        -6.917,   0.   ,  -5.618,  -5.716,   0.521,   3.354,  -7.528,
        -0.   ,  -6.646, -10.212,  -6.346,  -2.95 ,  -9.559,   0.327,
         0.56 ,  11.006,  -6.666,  -1.158,  -2.25 ,  -3.53 ,   9.035,
        -0.   ,  -2.316,  -2.   ,  -1.124,   0.804, -20.861,  27.166,
         0.   ,  -1.51 ,  -6.745,  -5.098,  15.455,   0.   ,   0.   ,
         0.629,  -3.

In [64]:
print(np.sum(np.abs(r.coef_)))
print(np.sum(np.abs(las001.coef_)))

print(np.sum(r.coef_ != 0))
print(np.sum(las001.coef_ != 0))

795.6521694490916
436.26164263065647
104
89


**Kesimpulan:** Ridge tidak membuat koefisien apa pun menjadi 0. Selain itu, pada dataset ini Lasso memberikan regularisasi keseluruhan yang lebih kuat daripada Ridge untuk nilai `alpha` tersebut (belum tentu berlaku secara umum).

In [65]:
y_pred = r.predict(X_pf_ss)
print(r2_score(y, y_pred))

y_pred = las001.predict(X_pf_ss)
print(r2_score(y, y_pred))

0.907527834057458
0.9102933722688233


**Kesimpulan**: Mengabaikan masalah overfitting, Ridge sedikit lebih baik daripada Lasso ketika `alpha` diatur 0,001 untuk keduanya (belum tentu berlaku secara umum).

# Contoh: Apakah Waktu Scaling Berpengaruh?

In [66]:
X_train, X_test, y_train, y_test = train_test_split(X_ss, y, test_size=0.3, 
                                                    random_state=72018)

In [67]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
r2_score(y_pred, y_test)

0.5832942094971805

In [68]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                    random_state=72018)

In [69]:
s = StandardScaler()
lr_s = LinearRegression()
X_train_s = s.fit_transform(X_train)
lr_s.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred_s = lr_s.predict(X_test_s)
r2_score(y_pred_s, y_test)

0.5832942094971799

**Kesimpulan:** Untuk Regresi Linear, tidak ada bedanya apakah scaling dilakukan sebelum atau sesudah dalam hal prediksi mentah. Namun hal ini berpengaruh pada algoritma lain. Selain itu, seperti yang akan kita lihat nanti, scaling bisa dijadikan bagian dari `Pipeline`.

# Bagaimana dengan Lasso?

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X_ss, y, test_size=0.3, 
                                                    random_state=72018)

In [80]:
lasso = Lasso()
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_test)
r2_score(y_pred, y_test)

0.2067158380521159

In [81]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                    random_state=72018)

In [82]:
s = StandardScaler()
lasso = Lasso()
X_train_s = s.fit_transform(X_train)
lasso.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred_s = lasso.predict(X_test_s)
r2_score(y_pred_s, y_test)

0.20880736283132462

Ya, ada perbedaan ketika standardisasi dilakukan setelah data dibagi, dan akurasinya menjadi lebih tinggi.